# שבוע 10: ניתוח כלי חרס — כד Ogame

בשיעור זה נלמד:
- כיצד לעבוד עם נתוני EFA של כלי חרס
- כיצד לנתח קשרי צורה-הקשר
- כיצד להשוות בין כבשנים שונים
- הכנה לפרויקט הסופי

> **הוראות**: מחברת זו **בנויה מראש** — הריצו תא אחר תא ללא כתיבת קוד חדש. התמקדו בפרשנות התוצאות.

In [ ]:
!pip install pyefd python-bidi -q
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

## מאגר הנתונים: כד Ogame

מאגר הנתונים (Wilczek et al. 2014) כולל כלי חרס מתקופת הברזל השני מאתרים שונים בצרפת.
כל פרט כולל:
- קו מתאר הכד (מחצית הפרופיל, נרמל ל-300 נקודות)
- מקדמי EFA
- משתני מטה-דאטה: הקשר (domestic / funerary / storage), כבשן, אתר

In [ ]:
import urllib.request

def load_csv_with_meta(url, meta_cols):
    """טעינת CSV עם עמודות מטה-דאטה ועמודות מספריות"""
    with urllib.request.urlopen(url) as r:
        lines = r.read().decode('utf-8').strip().split('\n')
    header = lines[0].split(',')
    meta, data = {c: [] for c in meta_cols}, []
    for line in lines[1:]:
        parts = line.strip().split(',')
        row_dict = dict(zip(header, parts))
        for c in meta_cols:
            meta[c].append(row_dict.get(c, 'unknown'))
        nums = [float(v) for k, v in row_dict.items() if k not in meta_cols]
        data.append(nums)
    return np.array(data), {k: np.array(v) for k, v in meta.items()}

base = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/pottery/'
try:
    efa_data, meta = load_csv_with_meta(base + 'ogame_efa.csv', ['context', 'kiln', 'site'])
    contexts = meta['context']
    kilns = meta['kiln']
    print(f'נטענו {len(efa_data)} כלי חרס, {efa_data.shape[1]} מקדמי EFA')
    print(f'הקשרים: {dict(zip(*np.unique(contexts, return_counts=True)))}')
    print(f'כבשנים: {dict(zip(*np.unique(kilns, return_counts=True)))}')
except Exception as e:
    print(f'משתמשים בנתוני דוגמה: {e}')
    np.random.seed(42)
    n = 90
    ctx_list = np.random.choice(['domestic', 'funerary', 'storage'], n, p=[0.5, 0.3, 0.2])
    kiln_list = np.random.choice(['FR', 'KM', 'unknown'], n, p=[0.4, 0.35, 0.25])
    # FR kilns are slightly different in shape (shift in EFA coefficients)
    base_coeff = np.random.randn(40) * 0.1
    efa_rows = []
    for i in range(n):
        shift = 0.2 if kiln_list[i] == 'FR' else (0.1 if kiln_list[i] == 'KM' else 0)
        efa_rows.append(base_coeff + np.random.randn(40)*0.1 + shift)
    efa_data = np.array(efa_rows)
    contexts = ctx_list
    kilns = kiln_list
    print(f'נוצרו {n} כלי חרס לדוגמה')

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
scores = pca.fit_transform(efa_data)
var = pca.explained_variance_ratio_ * 100

print('שונות מוסברת:')
for i in range(5):
    print(f'  PC{i+1}: {var[i]:.1f}%')

In [ ]:
unique_ctx = np.unique(contexts)
unique_kilns = np.unique(kilns)
palette_ctx = plt.cm.Set1(np.linspace(0, 0.8, len(unique_ctx)))
palette_kiln = plt.cm.Set2(np.linspace(0, 0.8, len(unique_kilns)))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# גרף לפי הקשר
ax = axes[0]
for ctx, color in zip(unique_ctx, palette_ctx):
    mask = contexts == ctx
    ax.scatter(scores[mask, 0], scores[mask, 1], c=[color], s=70, alpha=0.8,
               label=rtl(f'{ctx} (n={mask.sum()})'), edgecolors='white')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(rtl(f'PC1 ({var[0]:.1f}% שונות)'))
ax.set_ylabel(rtl(f'PC2 ({var[1]:.1f}% שונות)'))
ax.set_title(rtl('מרחב צורות כד Ogame — לפי הקשר'), fontsize=12)
ax.legend(fontsize=9)

# גרף לפי כבשן
ax = axes[1]
for kiln, color in zip(unique_kilns, palette_kiln):
    mask = kilns == kiln
    ax.scatter(scores[mask, 0], scores[mask, 1], c=[color], s=70, alpha=0.8,
               label=rtl(f'{kiln} (n={mask.sum()})'), edgecolors='white')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(rtl(f'PC1 ({var[0]:.1f}% שונות)'))
ax.set_ylabel(rtl(f'PC2 ({var[1]:.1f}% שונות)'))
ax.set_title(rtl('השוואת כבשנים — FR לעומת KM'), fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('pottery_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('הגרף נשמר: pottery_pca.png')

## בדיקה סטטיסטית: האם הקשר או כבשן מנבא צורה?

In [ ]:
def permutation_manova(X, groups, n_perm=999, seed=42):
    np.random.seed(seed)
    def f_stat(X, g):
        unique_g = np.unique(g)
        gm = X.mean(axis=0)
        between = sum(np.sum(g==u) * np.sum((X[g==u].mean(0) - gm)**2) for u in unique_g)
        within  = sum(np.sum((X[g==u] - X[g==u].mean(0))**2) for u in unique_g)
        return between / within if within > 0 else 0
    obs = f_stat(X, groups)
    perm = [f_stat(X, np.random.permutation(groups)) for _ in range(n_perm)]
    p = (np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1)
    return obs, p, obs / (obs + 1)

f1, p1, r1 = permutation_manova(scores[:, :4], contexts)
f2, p2, r2 = permutation_manova(scores[:, :4], kilns)

print('תוצאות MANOVA בפרמוטציה:')
print(f'  הקשר  → F={f1:.3f}, p={p1:.3f}, R²={r1:.3f}')
print(f'  כבשן  → F={f2:.3f}, p={p2:.3f}, R²={r2:.3f}')
print()
print('פרשנות:')
for name, p, r in [('הקשר', p1, r1), ('כבשן', p2, r2)]:
    sig = 'מובהק (p < 0.05)' if p < 0.05 else 'לא מובהק'
    print(f'  {name}: {sig}, R²={r:.3f} ({r*100:.1f}% שונות)')

## תרגיל — הכנה לפרויקט הסופי

1. איזה משתנה — הקשר או כבשן — מסביר יותר שונות בצורה?
2. מה המשמעות הארכאולוגית של תוצאה זו?
3. רשמו שאלת מחקר אחת ניתנת לבחינה עבור הפרויקט הסופי שלכם:
   - מאגר נתונים: ___
   - שאלה: ___
   - שיטה: ___
   - תוצאה צפויה: ___